In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"   # see issue #152
os.environ["CUDA_VISIBLE_DEVICES"]="6"

In [3]:
from aidan_lib.models.sam3_base import SAM3HarnessClient

/scratch4/home/adempst/projects/aidan-lib/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/scratch4/home/adempst/projects/aidan-lib/.venv/lib/python3.12/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [4]:
from pathlib import Path
import cv2
import imageio
from PIL import Image
import numpy as np

In [5]:
harness = SAM3HarnessClient(
    max_num_frames=120,
    max_frame_height=1080,
    max_frame_width=1920,
    address=("localhost", 26000),
    frame_dtype=np.uint8,
    shared_frame_memory_name="sam3_frames",
    shared_segmentation_memory_name="sam3_segmentations"
)

In [6]:
from aidan_lib.definitions import DATA_DIR
# test_vid_path = DATA_DIR / "tip_to_tip_short.mp4"
test_vid_path = DATA_DIR / "tip_to_tip.mp4"
assert test_vid_path.exists(), f"Test video does not exist {test_vid_path.absolute().as_posix()}"

In [7]:
from aidan_lib.video_utils.load_batched_frames import load_batched_frames, load_constrained_batched_frames
from aidan_lib.video_utils.scene_split import get_constrained_scenes, get_transnet_model

In [8]:
transnet = get_transnet_model("cuda")
constrained_scenes = get_constrained_scenes(test_vid_path, transnet, threshold=0.75)

/scratch4/home/adempst/projects/aidan-lib/.venv/lib/python3.12/site-packages/torch/nn/modules/linear.py:134: UserWarning: Deterministic behavior was enabled with either `torch.use_deterministic_algorithms(True)` or `at::Context::setDeterministicAlgorithms(true)`, but this operation is not deterministic because it uses CuBLAS and you have CUDA >= 10.2. To enable deterministic behavior in this case, you must set an environment variable before running your PyTorch application: CUBLAS_WORKSPACE_CONFIG=:4096:8 or CUBLAS_WORKSPACE_CONFIG=:16:8. For more information, go to https://docs.nvidia.com/cuda/cublas/index.html#results-reproducibility (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:304.)
  return F.linear(input, self.weight, self.bias)
/scratch4/home/adempst/projects/aidan-lib/.venv/lib/python3.12/site-packages/transnetv2_pytorch/transnetv2_pytorch.py:706: UserWarning: Deterministic behavior was enabled with either `torch.use_deterministic_algorithms(True)` or `at::Context

In [9]:
# batch_frame_loader = load_constrained_batched_frames(test_vid_path, constrained_scenes, batch_size=120, skip_frames=3, convert_pil=True, overlap=1)

In [10]:
# frame_batch, frame_numbers, done = next(batch_frame_loader)
# print(len(frame_batch))
# print(f"{frame_numbers[0]} to {frame_numbers[-1]}")
# print(done)

In [11]:
# out = harness("Person", frame_batch, prompt_frame=0, frame_numbers=frame_numbers, offload_state_to_cpu=None)

In [12]:
from IPython.display import Image as IPyImage
from IPython.display import display

In [13]:
from aidan_lib.visualization.segmentations import visualize_segmentations, int_mask_to_binary_masks

In [14]:
# test_frame_idx = 0
# test_global_frame_index = out.video_frame_indices[test_frame_idx]

# last_frame_img = frame_batch[test_frame_idx]
# sam_seg = out.segmentation[test_frame_idx]

# masks, obj_ids = int_mask_to_binary_masks(sam_seg, background_index=out.background_index)

# visualize_segmentations(last_frame_img, masks, labels=[f"Person {obj_ids[0]}"])

In [15]:
# frame_batch_2, frame_numbers_2, done_2 = next(batch_frame_loader)
# print(len(frame_batch_2))
# print(f"{frame_numbers_2[0]} to {frame_numbers_2[-1]}")
# print(done_2)

In [16]:
# out_2 = harness("Person", frame_batch_2, prompt_frame=0, frame_numbers=frame_numbers_2, offload_state_to_cpu=None)

In [17]:
# test_frame_idx = 6
# test_global_frame_index = out_2.video_frame_indices[test_frame_idx]

# last_frame_img = frame_batch_2[test_frame_idx]
# sam_seg = out_2.segmentation[test_frame_idx]

# masks, obj_ids = int_mask_to_binary_masks(sam_seg, background_index=out_2.background_index)

# if len(obj_ids) > 0:
#     img = visualize_segmentations(last_frame_img, masks, labels=[f"Person {obj_ids[0]}"])
# else:
#     print("Nobody visible")
#     img = last_frame_img
# img

In [18]:
from aidan_lib.models.sam3_base import SAM3VideoOutput
from typing import Iterable
def computer_overlap_ids(overlap_frames: list[int], last_out: SAM3VideoOutput, cur_out: SAM3VideoOutput, iou_thresh=0.9) -> list[tuple[int, int]]:
    """
    Returns a list of equality tuples.
    (last_obj_id, cur_obj_id) where we know that these refer to the same object
    """

    equality_counts = {}
    for global_frame in overlap_frames:
        last_frame_index = last_out.video_frame_indices.index(global_frame)
        cur_frame_index = cur_out.video_frame_indices.index(global_frame)

        last_seg = last_out.segmentation[last_frame_index]
        cur_seg = cur_out.segmentation[cur_frame_index]

        last_visible_obj_ids = [obj_id for obj_id in np.unique(last_seg) if obj_id != last_out.background_index]
        cur_visible_obj_ids = [obj_id for obj_id in np.unique(cur_seg) if obj_id != cur_out.background_index]

        print(last_visible_obj_ids, cur_visible_obj_ids)

        for last_obj_id in last_visible_obj_ids:
            last_mask = last_seg == last_obj_id
            for cur_obj_id in cur_visible_obj_ids:
                key = (int(last_obj_id), int(cur_obj_id))
                if key not in equality_counts:
                    equality_counts[key] = 0

                cur_mask = cur_seg == cur_obj_id

                intersection = np.count_nonzero(last_mask & cur_mask)
                # print(f"Intersection for {last_obj_id} and {cur_obj_id} is {intersection}")
                
                if intersection == 0:
                    # Then this can't be an equality
                    continue

                union = np.count_nonzero(last_mask | cur_mask)
                iou = intersection / union
                # print(f"IOU for {last_obj_id} and {cur_obj_id} is {iou}")

                if iou > iou_thresh:
                    equality_counts[key] += 1

    equalities = []
    for key, equality_count in equality_counts.items():
        if equality_count == len(overlap_frames):
            equalities.append(key)

    return equalities

In [19]:
from typing import NamedTuple

class FrameSegmentationInfo(NamedTuple):
    global_frame_num: int
    frame: np.ndarray
    segmentation: np.ndarray
    background_index: int

def generate_video_segmentation(batch_frame_loader):
    last_frame_numbers_set: set | None = None
    last_out: SAM3VideoOutput | None = None
    next_unique_id = 0
    last_obj_id_unique_id_assignments: dict | None = None

    last_global_frame = -1
    for frame_batch, frame_numbers, scene_done in batch_frame_loader:
        new_frame_numbers_set = set(frame_numbers)

        out = harness(
            "Person", frame_batch, frame_numbers=frame_numbers, offload_state_to_cpu=None
        )

        if last_frame_numbers_set:
            overlap = last_frame_numbers_set.intersection(new_frame_numbers_set)
            print(f"Overlaps {overlap}")
            assert last_out is not None
            equalities = computer_overlap_ids(list(overlap), last_out, out)
            print(f"Equalities {equalities}")
            cur_obj_id_to_prev_obj_id = {cur_obj_id: last_obj_id for (last_obj_id, cur_obj_id) in equalities}
        else:
            cur_obj_id_to_prev_obj_id = {}

        # Now we need to assign the object ids to unique ids
        seg = out.segmentation
        obj_ids = [int(obj_id) for obj_id in np.unique(seg) if obj_id != out.background_index]
        obj_id_to_unique_id_assignments = {}
        for cur_obj_id in obj_ids:
            # First, we see if there is a match to an old segmentation
            last_obj_id = cur_obj_id_to_prev_obj_id.get(cur_obj_id, None)
            if last_obj_id is None:
                unique_id = next_unique_id
                next_unique_id += 1
            else:
                assert last_obj_id_unique_id_assignments is not None
                unique_id = last_obj_id_unique_id_assignments[last_obj_id]
            obj_id_to_unique_id_assignments[cur_obj_id] = unique_id

        max_obj_id = int(np.max(seg))
        
        # Create a lookup table initialized to map to itself by default
        lookup_table = np.arange(max_obj_id + 1, dtype=np.int32)
        
        # Populate the lookup table with the dictionary assignments
        for old_id, new_id in obj_id_to_unique_id_assignments.items():
            lookup_table[old_id] = new_id

        for i in range(len(frame_batch)):
            global_frame_num = frame_numbers[i]
            if global_frame_num == last_global_frame:
                print("Dedupping overlap frame")
                continue
            last_global_frame = global_frame_num
            
            frame = frame_batch[i]
            frame_seg = out.segmentation[i]
            
            # Map the segmentation which uses obj ids to unique ids 
            # This applies the mapping to the entire 2D mask instantly
            unique_frame_seg = lookup_table[frame_seg]

            yield FrameSegmentationInfo(global_frame_num, frame, unique_frame_seg, out.background_index)

        if scene_done:
            last_frame_numbers_set = None
            last_out = None
            last_obj_id_unique_id_assignments = None
        else:
            last_frame_numbers_set = new_frame_numbers_set
            last_out = out
            last_obj_id_unique_id_assignments = obj_id_to_unique_id_assignments


In [20]:
batch_frame_loader = load_constrained_batched_frames(test_vid_path, constrained_scenes, batch_size=120, skip_frames=5, convert_pil=True, overlap=3)
frame_seg_generator = generate_video_segmentation(batch_frame_loader)

In [21]:
# frame_info = next(frame_seg_generator)

In [22]:
# frame_num, frame, sam_seg, background_index = frame_info

# masks, obj_ids = int_mask_to_binary_masks(sam_seg, background_index=background_index)

# img = visualize_segmentations(frame, masks, labels=[f"Person {obj_ids[0]}"])
# print(frame_num)
# display(img)

In [23]:
output_path = test_vid_path.parent / f"{test_vid_path.stem}_w_segs.mp4" 
print(output_path)

fps = 15
from tqdm import tqdm

# Use imageio's writer in a context manager to ensure it closes properly
progress = tqdm()
with imageio.get_writer(output_path, fps=fps, format='mp4', codec='libx264') as writer:
    for frame_info in frame_seg_generator:
        # Unpack the FrameSegmentationInfo
        frame_num, frame, sam_seg, background_index = frame_info
        
        # Convert segmentation to binary masks
        masks, obj_ids = int_mask_to_binary_masks(sam_seg, background_index=background_index)
        
        # Generate labels dynamically for however many objects are in the frame
        labels = [f"Person {obj_id}" for obj_id in obj_ids]
        
        # Create the visualization (img is a PIL Image)
        img = visualize_segmentations(frame, masks, labels=labels)
        
        # Convert the PIL Image to a NumPy array for imageio
        frame_array = np.expand_dims(np.array(img), axis=0)
        # print(frame_array.shape)
        
        # Write the frame to the video file
        writer.append_data(frame_array)
        
        # print(f"Processed and wrote frame: {frame_num}")
        progress.update(1)
        progress.set_description(f"Frame {frame_num}")

print(f"Video saved successfully to {output_path.absolute()}")

/z/home/adempst/data/aidan_lib/tip_to_tip_w_segs.mp4


Frame 14375: : 2876it [13:06,  5.73it/s]

Overlaps {14370, 14365, 14375}
[np.int8(0), np.int8(1), np.int8(2)] [np.int8(1)]
[np.int8(0), np.int8(1), np.int8(2)] [np.int8(1)]
[np.int8(0), np.int8(1), np.int8(2)] [np.int8(1)]
Equalities [(1, 1)]


Frame 16665: : 3337it [15:37,  8.05it/s]

Overlaps {16665, 16660, 16655}
[np.int8(0), np.int8(1)] [np.int8(0), np.int8(1)]
[np.int8(0), np.int8(1)] [np.int8(0), np.int8(1)]
[np.int8(0), np.int8(1)] [np.int8(0), np.int8(1)]
Equalities [(0, 0), (1, 1)]


...

Overlaps {86720, 86715, 86725}
[np.int8(0), np.int8(3)] [np.int8(0), np.int8(1)]
[np.int8(0), np.int8(3)] [np.int8(0), np.int8(1)]
[np.int8(0), np.int8(3)] [np.int8(0), np.int8(1)]
Equalities [(0, 1), (3, 0)]


Frame 87195: : 17578it [1:33:30, 10.73it/s]

Overlaps {87185, 87195, 87190}
[np.int8(6)] [np.int8(0)]
[np.int8(6)] [np.int8(0), np.int8(1)]
[np.int8(6)] [np.int8(0)]
Equalities [(6, 0)]


Frame 89300: : 18002it [1:36:50,  3.35it/s]

Video saved successfully to /z/home/adempst/data/aidan_lib/tip_to_tip_w_segs.mp4
